# 4.6 暂退法（Dropout）

## 1. 核心思想

训练时随机丢弃部分神经元，防止过拟合并提升泛化。标准暂退法定义：

$$
h' =
\begin{cases}
0 & \text{概率 } p \\
\frac{h}{1-p} & \text{其他情况}
\end{cases}
$$

**我如何理解dropout：**
1. 视为训练多个子网络的集合，每次训练随机选择一个子网络（但后来学术界并不完全认可这种解释）。
2. 强迫模型不能过度依赖特定神经元。
3. 每次迭代只更新部分参数，使得更新更加保守，减少过度拟合风险。
4. 相当于在隐藏层激活上添加了噪声，提升模型鲁棒性。
5. 与权重衰退（L2正则化）效果类似。
6. 调参常用方法：发现64适合这个数据集，使用128*0.5效果好于64。

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l
%run ../my_utils.py

## 2. 从零开始实现

先实现单层的 `dropout_layer`。
- `mask = (torch.rand(X.shape) > dropout).float()`对于计算更高效。

In [ ]:
def dropout_layer(X, dropout):
    assert 0 <= dropout <= 1
    if dropout == 1:
        return torch.zeros_like(X)
    if dropout == 0:
        return X
    mask = (torch.rand(X.shape) > dropout).float()
    return mask * X / (1.0 - dropout)

简单测试：

In [ ]:
X = torch.arange(16, dtype=torch.float32).reshape((2, 8))
print(X)
print(dropout_layer(X, 0.0))
print(dropout_layer(X, 0.5))
print(dropout_layer(X, 1.0))

### 2.1 定义模型参数

使用 Fashion-MNIST，设置两个隐藏层。

In [ ]:
num_inputs, num_outputs, num_hiddens1, num_hiddens2 = 784, 10, 256, 256

### 2.2 定义模型

在训练时对隐藏层输出应用暂退法。

In [ ]:
dropout1, dropout2 = 0.2, 0.5

class Net(nn.Module):
    def __init__(self, num_inputs, num_outputs, num_hiddens1, num_hiddens2,
                 is_training=True):
        super(Net, self).__init__()
        self.num_inputs = num_inputs
        self.training = is_training
        self.lin1 = nn.Linear(num_inputs, num_hiddens1)
        self.lin2 = nn.Linear(num_hiddens1, num_hiddens2)
        self.lin3 = nn.Linear(num_hiddens2, num_outputs)
        self.relu = nn.ReLU()

    def forward(self, X):
        H1 = self.relu(self.lin1(X.reshape((-1, self.num_inputs))))
        if self.training == True:
            H1 = dropout_layer(H1, dropout1)
        H2 = self.relu(self.lin2(H1))
        if self.training == True:
            H2 = dropout_layer(H2, dropout2)
        out = self.lin3(H2)
        return out


net = Net(num_inputs, num_outputs, num_hiddens1, num_hiddens2)

### 2.3 训练和测试

In [ ]:
num_epochs, lr, batch_size = 10, 0.5, 256
loss = nn.CrossEntropyLoss(reduction='none')
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)
trainer = torch.optim.SGD(net.parameters(), lr=lr)
d2l.train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

## 3. 简洁实现

使用 `nn.Dropout` 直接插入到全连接层之后。

In [ ]:
net = nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 256),
        nn.ReLU(),
        # 在第一个全连接层之后添加一个 dropout 层
        nn.Dropout(dropout1),
        nn.Linear(256, 256),
        nn.ReLU(),
        # 在第二个全连接层之后添加一个 dropout 层
        nn.Dropout(dropout2),
        nn.Linear(256, 10))

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights);

In [ ]:
trainer = torch.optim.SGD(net.parameters(), lr=lr)
train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer)

## 小结

- 暂退法在训练时随机丢弃部分神经元，抑制过拟合
- 采用缩放保持期望不变，测试时不使用暂退
- 框架提供 `nn.Dropout`，可直接插入网络